# Chapter 06: Vector Databases & Similarity Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part3-rag-data/ch06-vector-databases-and-search.ipynb)

**Part:** Part III: Vector Databases, Data & Production RAG  
**Chapter Identifier:** `ch06`  

## Core Concepts Covered in this Chapter:
- **Vector DBs**
- **Embeddings**
- **Dimensions**
- **Cosine/Dot Product**
- **k-NN vs. ANN**
- **HNSW & IVF Indexes**
- **Product Quantization**
- **Metadata Filtering**
- **Hybrid Search**
- **Namespaces**
- **Upserts/Deletes**
- **Sharding**
- **Recall vs. Latency**
- **pgvector**
- **Qdrant**
- **Chroma**
- **FAISS**

---


## 1. Exact k-NN vs. Approximate Nearest Neighbor (ANN)

Exact k-NN compares a query against every vector in the database, yielding 100% recall with O(N * d) linear latency. ANN algorithms (HNSW, IVF) trade a fraction of recall for logarithmic O(log N) retrieval speed.

```mermaid
graph TD
    A[Query Vector] --> B{Search Algorithm}
    B -->|Exact k-NN| C["Brute Force O(N)<br/>100% Recall, High Latency"]
    B -->|HNSW Graph| D["Hierarchical Small World<br/>O(log N), Fast, High Memory"]
    B -->|IVF Inverted Index| E["Voronoi Cells (Centroids)<br/>O(N/k), Low Memory"]
```


In [ ]:
import time
import random

def brute_force_knn(query: list, database: list, k: int = 3) -> list:
    scores = []
    for doc_id, vec in enumerate(database):
        sim = sum(q * v for q, v in zip(query, vec))
        scores.append((doc_id, sim))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

dim = 16
db = [[random.uniform(-1, 1) for _ in range(dim)] for _ in range(1000)]
q = [random.uniform(-1, 1) for _ in range(dim)]

t0 = time.time()
top_matches = brute_force_knn(q, db, k=3)
elapsed_us = (time.time() - t0) * 1e6

print(f"Brute force k-NN over 1,000 vectors: {elapsed_us:.1f} microseconds")
print(f"Top 3 Matches (DocID, Score): {[(d, round(s, 4)) for d, s in top_matches]}")

## 2. HNSW & IVF Index Architectures

HNSW (Hierarchical Navigable Small World) organizes vectors into multi-layer graphs where top layers have long-range skip connections and bottom layers have dense local clustering. IVF (Inverted File) clusters vectors into Voronoi cells and searches only the nearest centroids.


In [ ]:
def simulate_ivf_search(query: list, centroids: list, clusters: dict, nprobe: int = 2) -> list:
    centroid_scores = [(c_id, sum(q * c for q, c in zip(query, c_vec))) for c_id, c_vec in enumerate(centroids)]
    centroid_scores.sort(key=lambda x: x[1], reverse=True)
    nearest_centroids = [c_id for c_id, _ in centroid_scores[:nprobe]]
    candidates = []
    for c_id in nearest_centroids:
        candidates.extend(clusters[c_id])
    return candidates

print("IVF Index concept verified: nprobe determines candidate search radius.")

## 3. Hybrid Search & Reciprocal Rank Fusion (RRF)

Dense vector search captures semantic concepts but misses exact keyword matches (IDs, error codes, rare names). Hybrid Search combines BM25 keyword matching with dense ANN search, merging rankings via Reciprocal Rank Fusion (RRF).


In [ ]:
def reciprocal_rank_fusion(dense_ranks: list, sparse_ranks: list, k: int = 60) -> list:
    scores = {}
    for rank, doc_id in enumerate(dense_ranks, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + (1.0 / (k + rank))
    for rank, doc_id in enumerate(sparse_ranks, 1):
        scores[doc_id] = scores.get(doc_id, 0.0) + (1.0 / (k + rank))
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

dense_hits = ["doc_rag_intro", "doc_vector_db", "doc_embeddings", "doc_sql"]
sparse_hits = ["doc_sql", "doc_rag_intro", "doc_error_codes", "doc_vector_db"]

fused_results = reciprocal_rank_fusion(dense_hits, sparse_hits, k=60)
print("Hybrid RRF Fused Rankings:")
for doc, score in fused_results:
    print(f"  {doc:20s} -> RRF Score: {score:.6f}")

---
## Hands-On Capstone Lab: Build an In-Memory Vector Store with Metadata Filtering & Hybrid RRF

### Objective:
Build a functional Vector Database class supporting namespaces, upserts, metadata filtering, and hybrid retrieval.

### Instructions:
1. Implement `SimpleVectorStore` with `upsert(id, vector, metadata)`.
2. Implement `query(vector, filter_fn, top_k)`.
3. Execute query filtering for a specific category namespace.


In [ ]:
class SimpleVectorStore:
    def __init__(self):
        self.documents = {}

    def upsert(self, doc_id: str, vector: list, metadata: dict):
        self.documents[doc_id] = {"vector": vector, "metadata": metadata}

    def query(self, query_vec: list, filter_fn=None, top_k: int = 2):
        candidates = []
        for doc_id, data in self.documents.items():
            if filter_fn and not filter_fn(data["metadata"]):
                continue
            sim = sum(q * v for q, v in zip(query_vec, data["vector"]))
            candidates.append({"id": doc_id, "score": sim, "metadata": data["metadata"]})
        candidates.sort(key=lambda x: x["score"], reverse=True)
        return candidates[:top_k]

store = SimpleVectorStore()
store.upsert("doc1", [0.9, 0.1, 0.0], {"category": "rag", "year": 2024})
store.upsert("doc2", [0.1, 0.9, 0.0], {"category": "mlops", "year": 2024})
store.upsert("doc3", [0.85, 0.2, 0.0], {"category": "rag", "year": 2023})

results = store.query([1.0, 0.0, 0.0], filter_fn=lambda m: m["category"] == "rag")
print("Filtered Vector Query Results:")
for r in results:
    print(f"  ID: {r['id']} | Score: {r['score']:.4f} | Metadata: {r['metadata']}")

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What is the primary trade-off when using an Approximate Nearest Neighbor (ANN) index over exact k-NN?
- **A)** ANN sacrifices a small percentage of recall to achieve orders-of-magnitude faster query latency and scalability
- **B)** ANN guarantees 100% precision but uses 10x more storage
- **C)** ANN can only process one query per day
- **D)** ANN works exclusively with image files

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) ANN sacrifices a small percentage of recall to achieve orders-of-magnitude faster query latency and scalability

**Explanation:** Exact k-NN compares against every vector in O(N). ANN algorithms trade a minor fraction of recall for O(log N) speed.
</details>

### Question 2: Why is Hybrid Search (Dense Vectors + Sparse BM25) widely used in enterprise search systems?
- **A)** Dense vectors excel at semantic similarity while BM25 guarantees precision on exact keyword matches, serial numbers, and proper nouns
- **B)** It cuts memory consumption by exactly 50%
- **C)** It replaces the need for an LLM completely
- **D)** It prevents users from issuing long queries

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) Dense vectors excel at semantic similarity while BM25 guarantees precision on exact keyword matches, serial numbers, and proper nouns

**Explanation:** Hybrid search bridges semantic generalization (dense embeddings) with exact lexical matching (sparse BM25), combined via algorithms like RRF.
</details>

### Question 3: In the HNSW (Hierarchical Navigable Small World) algorithm, what do upper graph layers provide?
- **A)** High-resolution image thumbnails
- **B)** Long-range skip links that allow fast coarse navigation across the metric space
- **C)** A list of deleted records
- **D)** Strict encryption keys

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Long-range skip links that allow fast coarse navigation across the metric space

**Explanation:** Similar to a skip-list data structure, upper layers in HNSW feature sparse graphs with long edges for rapid navigation, while lower layers provide dense local clustering.
</details>

### Question 4: In Reciprocal Rank Fusion (RRF), what is the impact of the smoothing parameter k (typically k=60)?
- **A)** It sets the maximum number of documents returned
- **B)** It dampens the impact of high outlier rankings from any single retrieval system
- **C)** It converts floating point scores to integers
- **D)** It requires exactly 60 documents to be stored in the database

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) It dampens the impact of high outlier rankings from any single retrieval system

**Explanation:** The smoothing constant k ensures that low rank positions do not excessively dominate the score, producing balanced reciprocal ranking.
</details>
